# Plant Disease Dataset Loader using TensorFlow

This notebook loads potato and tomato disease images using TensorFlow's `tf.data` API with automatic batching, data augmentation, and proper train/validation/test splitting.

**Dataset**: 13 classes - 3 Potato + 10 Tomato
**Image Size**: 256x256
**Batch Size**: 32
**Split**: 80% train, 10% validation, 10% test

## Step 1: Setup and Imports

In [ ]:
import matplotlib
matplotlib.use('Agg')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing import image_dataset_from_directory
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## Step 2: Define Dataset Path and Configuration

In [ ]:
# Define dataset configuration
DATASET_PATH = "/Users/muddsarbutt/Documents/Claude/DiseaseDetection/training/dataset"
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
SEED = 42

# Verify dataset exists
dataset_path = Path(DATASET_PATH)
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset path not found: {DATASET_PATH}")

# List classes
classes = sorted([d.name for d in dataset_path.iterdir() if d.is_dir()])
print(f"Dataset path: {DATASET_PATH}")
print(f"Classes found: {classes}")
print(f"Number of classes: {len(classes)}")

# Count images per class
for cls in classes:
    class_path = dataset_path / cls
    image_count = len(list(class_path.glob("*.JPG"))) + len(list(class_path.glob("*.jpg"))) + len(list(class_path.glob("*.png")))
    print(f"  {cls}: {image_count} images")

## Step 3: Load Full Dataset using tf.data API

In [ ]:
# Load the full dataset with automatic class discovery
full_dataset = image_dataset_from_directory(
    DATASET_PATH,
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print(f"Full dataset loaded")
print(f"Class names: {full_dataset.class_names}")
print(f"Number of batches: {len(full_dataset)}")

## Step 4: Split Dataset into Train (80%) and Temp (20%)

In [ ]:
# Calculate split sizes
total_batches = len(full_dataset)
train_size = int(0.8 * total_batches)
temp_size = total_batches - train_size

print(f"Total batches: {total_batches}")
print(f"Train batches (80%): {train_size}")
print(f"Temp batches (20%): {temp_size}")

# Split into train and temp (validation + test)
train_dataset = full_dataset.take(train_size)
temp_dataset = full_dataset.skip(train_size)

print(f"\nTrain dataset batches: {len(train_dataset)}")
print(f"Temp dataset batches: {len(temp_dataset)}")

## Step 5: Split Temp into Validation (10%) and Test (10%)

In [ ]:
# Split temp into validation and test (1:1 split)
val_size = int(0.5 * temp_size)

val_dataset = temp_dataset.take(val_size)
test_dataset = temp_dataset.skip(val_size)

print(f"Validation dataset batches: {len(val_dataset)}")
print(f"Test dataset batches: {len(test_dataset)}")

# Calculate approximate image counts
approx_train_images = train_size * BATCH_SIZE
approx_val_images = val_size * BATCH_SIZE
approx_test_images = (len(test_dataset)) * BATCH_SIZE
total_images = approx_train_images + approx_val_images + approx_test_images

print(f"\nApproximate image counts:")
print(f"  Train: {approx_train_images} ({100*approx_train_images/total_images:.1f}%)")
print(f"  Validation: {approx_val_images} ({100*approx_val_images/total_images:.1f}%)")
print(f"  Test: {approx_test_images} ({100*approx_test_images/total_images:.1f}%)")
print(f"  Total: {int(total_images)}")

## Step 6: Apply Performance Optimizations

In [ ]:
# Apply cache, shuffle, and prefetch for performance optimization
SHUFFLE_BUFFER = 1000

train_dataset = train_dataset.cache().shuffle(SHUFFLE_BUFFER).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.cache().shuffle(SHUFFLE_BUFFER).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.cache().shuffle(SHUFFLE_BUFFER).prefetch(tf.data.AUTOTUNE)

print("Performance optimizations applied:")
print(f"  ✓ Caching enabled for all datasets")
print(f"  ✓ Shuffling enabled (buffer_size={SHUFFLE_BUFFER})")
print(f"  ✓ Prefetching enabled (tf.data.AUTOTUNE)")

## Step 7: Build Transfer Learning Model (MobileNetV2)

In [ ]:
INPUT_SHAPE = (IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
N_CLASSES = len(full_dataset.class_names)

# Data augmentation layers
data_augmentation = keras.Sequential([
    keras.layers.RandomRotation(0.2),
    keras.layers.RandomFlip("horizontal"),
])

# Load pre-trained MobileNetV2 (without top classification layer)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=INPUT_SHAPE,
    include_top=False,
    weights='imagenet'
)

# Freeze the base model weights
base_model.trainable = False

# Build the full model
model = keras.Sequential([
    # Preprocessing
    keras.layers.Resizing(IMAGE_SIZE[0], IMAGE_SIZE[1]),
    keras.layers.Rescaling(1./255),

    # Data augmentation (only active during training)
    data_augmentation,

    # Pre-trained feature extractor
    base_model,

    # Classification head
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(N_CLASSES, activation='softmax'),
])

print(f"Number of classes: {N_CLASSES}")
print(f"Classes: {full_dataset.class_names}")
print(f"Base model: MobileNetV2 (frozen, ImageNet weights)")
print(f"Base model params: {base_model.count_params():,}")
model.build(input_shape=(BATCH_SIZE, IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
model.summary()

## Step 8: Compile the Model

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

print("Model compiled:")
print("  Optimizer: Adam")
print("  Loss: SparseCategoricalCrossentropy")
print("  Metrics: Accuracy")

## Step 9: Train the Model

In [ ]:
EPOCHS = 15

history = model.fit(
    train_dataset,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=val_dataset,
)

print(f"\nTraining complete after {EPOCHS} epochs")

## Step 10: Evaluate the Model

In [ ]:
scores = model.evaluate(test_dataset)

print(f"\nTest Loss: {scores[0]:.4f}")
print(f"Test Accuracy: {scores[1]*100:.2f}%")

## Step 11: Predict on Sample Test Images

In [ ]:
class_names = full_dataset.class_names

# Get a batch from test dataset
test_images, test_labels = next(iter(test_dataset))

# Predict on first 3 images
predictions = model.predict(test_images[:3])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, ax in enumerate(axes):
    predicted_idx = np.argmax(predictions[i])
    actual_idx = test_labels[i].numpy()
    
    ax.imshow(test_images[i].numpy().astype("uint8"))
    color = "green" if predicted_idx == actual_idx else "red"
    ax.set_title(
        f"Actual: {class_names[actual_idx]}\nPredicted: {class_names[predicted_idx]}\nConfidence: {predictions[i][predicted_idx]*100:.1f}%",
        fontsize=10, fontweight='bold', color=color
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 12: Save the Model and Register in MLflow

In [ ]:
from mlflow_config import get_next_version, register_model

# Auto-detect next version
MODEL_VERSION = get_next_version()
model_path = f"models/potato_disease_model/{MODEL_VERSION}"

# Export model for TF Serving
model.export(model_path)
print(f"Model exported to: {model_path}")

# Register in MLflow
params = {
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE[0],
    "n_classes": N_CLASSES,
    "classes": ",".join(full_dataset.class_names),
}
version = register_model(model_path, history, params, keras_model=model)
print(f"MLflow version: {version}")